# Pulldown Y SNPs with OY SNPs

In [12]:
import numpy as np
import os  # For Saving to Folder
import pandas as pd
import matplotlib.pyplot as plt

import socket
import os as os
import sys as sys
import multiprocessing as mp
from pysam import AlignmentFile

### For Arial Font
from matplotlib import rcParams
rcParams['font.family'] = 'sans-serif'   # Set the defaul
### Make sure to have the font installed (it is on cluster for Harald)
rcParams['font.sans-serif'] = ['Arial']

socket_name = socket.gethostname()
print(socket_name)

if socket_name.startswith("compute-"):
    print("HSM Computational partition detected.")
    path = "/n/groups/reich/hringbauer/git/y_chrom/"  # The Path on Midway Cluster
    
elif socket_name.startswith("bionc") or socket_name.startswith("hpc"):
    print("Leipzig Cluster detected!")
    path = "/mnt/archgen/users/hringbauer/git/y_chrom/"
    
else:
    raise RuntimeWarning("Not compatible machine. Check!!")

os.chdir(path)  # Set the right Path (in line with Atom default)

# Show the current working directory. Should be HAPSBURG/Notebooks/ParallelRuns
print(os.getcwd())
print(f"CPU Count: {mp.cpu_count()}")
print(sys.version)

### Custom Imports
from python.pulldown import load_snp_file_ISOGG, call_y_bam, mismatch_path

hpc030
Leipzig Cluster detected!
/mnt/archgen/users/hringbauer/git/y_chrom
CPU Count: 128
3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]


# 0) Prepare Data

### 0a) Load Autorun Eager Dictionary mapping iids to bam files

In [2]:
dft = pd.read_csv("/mnt/archgen/users/hringbauer/git/auto_popgen/output/TF/v0.3/bam_paths.tsv", sep="\t")
dft2 = pd.read_csv("/mnt/archgen/users/hringbauer/git/auto_popgen/output/RM/v0.3/bam_paths.tsv", sep="\t")
n = np.sum(dft["bam#"]>0)
bam_dict = dict(zip(dft["iid"], dft["bam_path"]))
bam_dict2 = dict(zip(dft2["iid"], dft2["bam_path"]))
print(f"Loaded {len(dft)} Individuals. With BAM: {n}")
print(f"Loaded {len(dft2)} RM Individuals.")

Loaded 28636 Individuals. With BAM: 17215
Loaded 4607 RM Individuals.


### 0b) Prepare SNP List

In [3]:
df = load_snp_file_ISOGG("./data/all_snps.csv")

Index(['Name', 'Subgroup Name', 'Alternate Names', 'rs numbers',
       'Build 37 Number', 'Build 38 Number', 'Mutation Info'],
      dtype='object')
Loaded 92035 SNPs
# Positions available: 91881
# Biallelic SNPs: 91814
# Ref & Alt different: 91811
# Ref & Alt ACTG: 91806
# Unique SNP positions: 73148


In [28]:
df1 = pd.read_csv("/mnt/archgen/users/hringbauer/git/y_chrom/data/all_snps_filtered_levels.csv", low_memory=False)
print(f"Loaded {len(df1)} OY SNPs with levels loaded")

Loaded 2868884 OY SNPs with levels loaded


In [9]:
df1.head()

,Unnamed: 0,SNP-ID,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level
0,1557478,FTG48817,Y,16259936.0,G,T,NaN,NaN,0
1,1557470,MF630222,Y,16259923.0,T,G,NaN,NaN,0
2,1557471,FT415097,Y,16259925.0,C,T,NaN,NaN,0
3,1557472,FTE94401,Y,16259927.0,C,T,NaN,NaN,0
4,2527307,FTC61890,Y,22705984.0,G,C,NaN,NaN,0


### Create BED file for OY [1x requirement]

In [25]:
savepath = "./data/OY_snps.bed"

dft = df1.sort_values(by="pos")
dft = dft[["chrom", "pos"]].copy()

dft["pos1"] = dft["pos"]
dft.to_csv(savepath, sep="\t", index=False, header=None)
print(f"Saved {len(dft)} OY SNPs to {savepath}")

Saved 2868884 OY SNPs to ./data/OY_snps.bed


### 1) Run Y Calling

### 1a) Single Example with ISOGG SNPs

In [62]:
%%time
path_bam = bam_dict2["KKG002"]

df_ch, df_der = call_y_bam(df=df, 
                           path_bam=path_bam) #A55903 and A55904
len(df_der)

Average Coverage: 5.1489x
#Sites covered: 58158/73148
#Derived Loci: 
1030 / 58158 covered>0
CPU times: user 99.5 ms, sys: 2.7 ms, total: 102 ms
Wall time: 2.57 s


1030

In [63]:
df_der[-100:-50]

### This sample is J2a1a1b1a - looks like 2 SNPs are derived there

,Name,chrom,pos,ref,alt,Subgroup Name,Alternate Names,rs numbers,A,C,G,T,ref#,alt#
930,PF4965,Y,15610713,G,T,J2a,CTS4360,NaN,0,0,0,10,0,10
931,CTS4699,Y,15775488,A,G,J2a,PF4909,NaN,0,0,3,0,0,3
932,PF4966,Y,15704139,C,T,J2a,CTS4540,NaN,0,0,0,5,0,5
933,F841,Y,6854256,C,T,J2a,PF4948,NaN,0,0,0,5,0,5
934,CTS9538,Y,18952077,G,A,J2a,PF4919,NaN,1,0,0,0,0,1
935,L505,Y,21970721,G,T,J2a,PF4987,NaN,0,0,0,1,0,1
936,PF4953,Y,7680253,C,G,J2a,NaN,NaN,0,0,6,0,0,6
937,PF4952,Y,7676739,C,T,J2a,NaN,NaN,0,0,0,7,0,7
938,PF5112,Y,23033706,C,T,J2a1,CTS11251,NaN,0,0,0,1,0,1
939,PF4610,Y,22757708,G,C,J2a1,NaN,NaN,0,3,0,0,0,3


In [61]:
s = "J2a1a1b1a"
mismatch_path(s, df_ch).sort_values(by="Subgroup Name")[-50:]

Mismatches: 0 / 0


,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#


### 1b) Try OY SNPs

In [65]:
%%time
path_bam = bam_dict2["KKG002"]

df_ch, df_der = call_y_bam(df=df1, path_bam=path_bam,
                           path_bed='/mnt/archgen/users/hringbauer/git/y_chrom/data/OY_snps.bed') 
len(df_der)

Average Coverage: 1.3794x
#Sites covered: 907725/2868884
#Derived Loci: 
6981 / 907725 covered>0
CPU times: user 1.28 s, sys: 432 ms, total: 1.71 s
Wall time: 22.7 s


6981

In [66]:
df_der.sort_values(by="Level")[-50:]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
6617,Y6453,Y,8250772,G,A,R-Y6453,NaN,51,1,0,0,0,0,1
236,A39534,Y,18881152,G,A,R-FTF86763,NaN,51,1,0,0,0,0,1
3743,FTG77865,Y,18089187,G,A,N-FTG77668,NaN,51,1,0,0,0,0,1
7,A1234,Y,14975852,C,T,R-A1234,NaN,51,0,0,0,1,0,1
2383,FT157294,Y,15925461,G,A,J-FT157054,NaN,51,1,0,0,0,0,1
2981,FTA59569,Y,7510363,C,T,R-FTA59569,NaN,52,0,0,0,1,0,1
2971,FTA56456,Y,7834060,G,A,R-FTA81373,NaN,52,1,0,0,0,0,1
2571,FT278583,Y,22512323,G,A,J-BY55828,NaN,52,1,0,0,0,0,1
2724,FT378570,Y,23299264,G,A,R-FT377470,NaN,52,1,0,0,0,0,1
799,BY83284,Y,10036953,G,A,R-BY77186,NaN,52,1,0,0,0,0,1


In [67]:
dfc = df_der.groupby("Y-haplogroup").agg(
    n=('Y-haplogroup', 'count'),
    level=('Level', 'mean')).reset_index()
dfc2 = dfc[dfc["n"]>1] # Only extract Y haplogroups that have 2 SNPs derived

In [68]:
dfc2.sort_values(by="level")[-50:]

,Y-haplogroup,n,level
0,A-AF6,7,0.0
136,F-F21819,3,0.0
367,J-Page28,11,0.0
361,J-M304,2,0.0
447,O-F22068,2,0.0
617,R-BY74074,2,0.0
4,A-L1090,60,2.0
3,A-L1085,12,2.0
11,A-V168,5,3.0
17,A00-FGC26001,2,3.0


In [87]:
df_der[df_der["Y-haplogroup"]=="J-PF5172"]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#


In [85]:
df_der[df_der["Subgroup Name"]=="J-Y153763"]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#


In [81]:
df_der[df_der["Subgroup Name"].str.contains("PF5190")]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#


The Y-Haplogroup seems to be J-Y153763. It is two G->A, but the parent haplogroup, PF5172, also has two G-A derived.

# 2) Run Henry II

In [78]:
%%time
path_bam = "/mnt/archgen/Autorun_eager/eager_outputs/SG/BMG/BMG001/trimmed_bam/BMG001_ss_libmerged_udghalf.trimmed.bam"

df_ch, df_der = call_y_bam(df=df1, path_bam=path_bam,
                           path_bed='/mnt/archgen/users/hringbauer/git/y_chrom/data/OY_snps.bed') 

Average Coverage: 0.3605x
#Sites covered: 813297/2868884
#Derived Loci: 
3469 / 813297 covered>0
CPU times: user 1.23 s, sys: 308 ms, total: 1.54 s
Wall time: 19.2 s


3469

In [89]:
dfc = df_der.groupby("Y-haplogroup").agg(
    n=('Y-haplogroup', 'count'),
    level=('Level', 'mean')).reset_index()
dfc2 = dfc[dfc["n"]>1] # Only extract Y haplogroups that have 2 SNPs derived

In [102]:
dfc.sort_values(by="level")[-50:] R-FTA98646 and R-FTF30738

,Y-haplogroup,n,level
321,R-L196,1,46.0
258,R-FGC14769,1,46.0
312,R-FTE20211,1,46.0
207,R-BY103964,1,47.0
216,R-BY172590,1,47.0
242,R-BY77606,1,47.0
282,R-FT371843,1,47.0
252,R-BY99994,1,47.0
247,R-BY95411,1,47.0
238,R-BY68473,1,47.0


In [95]:
df_ch[df_ch["Subgroup Name"].str.contains("FTA63331")]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
844707,FTA63331,Y,8840886,G,T,R-FTA63331,NaN,46,0,0,0,1,0,1


In [112]:
df_ch[df_ch["Y-haplogroup"]=="R-BY78034"] # -FTA63879

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
868616,BY78034,Y,9096231,G,A,R-BY78034,NaN,54,1,0,0,0,0,1
868722,FT412112,Y,23446881,A,G,R-BY78034,NaN,54,1,0,0,0,1,0
